# 01E — FINAL Global Image-Family Reconstruction

**NO MODEL TRAINING.**

Why this notebook exists:
- 01B found additional relationships after the first cleanup.
- 01D found additional relationships after the second cleanup.
- That happened because the earlier audits kept only a nearest candidate per image.

This notebook instead performs a **global all-pairs pHash candidate search** over the complete V2 dataset,
then verifies every suspicious candidate with SIFT + RANSAC and builds connected image families.

Output:
`Cataract/Data_Clean_LeakageControlled_FINAL`

The original `Cataract/Data`, V1, and V2 folders are never modified.

In [2]:
# ============================================================
# CELL 1 — SETUP AND GOOGLE DRIVE PATHS
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import numpy as np
import shutil
import json
import hashlib
import os

PROJECT = Path('/content/drive/MyDrive/Cataract')

SOURCE = PROJECT / 'Data_Clean_LeakageControlled_v2'

FINAL = PROJECT / 'Data_Clean_LeakageControlled_FINAL'

REPORT = (
    PROJECT
    / 'FINAL_REVISION_2026_08'
    / 'global_family_audit'
)

QUAR = (
    PROJECT
    / 'FINAL_REVISION_2026_08'
    / 'global_family_quarantine'
)

CLASS_ORDER = [
    'Cataract',
    'Normal',
    'Not Eye'
]

SPLITS = [
    'Train',
    'Validation',
    'Test'
]

SEED = 42

REPORT.mkdir(parents=True, exist_ok=True)

print("PROJECT:", PROJECT)
print("SOURCE :", SOURCE)
print("FINAL  :", FINAL)
print("REPORT :", REPORT)
print("QUAR   :", QUAR)

print("\nChecking important folders...")

print("SOURCE exists:", SOURCE.exists())
print("REPORT exists:", REPORT.exists())

assert SOURCE.exists(), f"STOP: Cannot find {SOURCE}"

print("\n✅ CELL 1 COMPLETE")

Mounted at /content/drive
PROJECT: /content/drive/MyDrive/Cataract
SOURCE : /content/drive/MyDrive/Cataract/Data_Clean_LeakageControlled_v2
FINAL  : /content/drive/MyDrive/Cataract/Data_Clean_LeakageControlled_FINAL
REPORT : /content/drive/MyDrive/Cataract/FINAL_REVISION_2026_08/global_family_audit
QUAR   : /content/drive/MyDrive/Cataract/FINAL_REVISION_2026_08/global_family_quarantine

Checking important folders...
SOURCE exists: True
REPORT exists: True

✅ CELL 1 COMPLETE


## Phase 1 — Fingerprint every V2 image

This creates:
- SHA-256
- 64-bit perceptual hash
- class
- old V2 split

The fingerprint table is cached in Drive, so rerunning the notebook does not repeat this phase.

In [3]:
# ============================================================
# CELL 2 — LOAD SAVED GLOBAL ANALYSIS
# ============================================================

family_path = REPORT / 'global_family_membership.csv'

confirmed_path = (
    REPORT
    / 'GLOBAL_CONFIRMED_NEAR_DUPLICATE_EDGES.csv'
)

print("Checking saved analysis files...")

print(
    "global_family_membership.csv:",
    family_path.exists()
)

print(
    "GLOBAL_CONFIRMED_NEAR_DUPLICATE_EDGES.csv:",
    confirmed_path.exists()
)

assert family_path.exists(), (
    "STOP: global_family_membership.csv is missing."
)

assert confirmed_path.exists(), (
    "STOP: GLOBAL_CONFIRMED_NEAR_DUPLICATE_EDGES.csv is missing."
)

df = pd.read_csv(family_path)

confirmed = pd.read_csv(confirmed_path)

print("\nLoaded saved analysis.")

print("Family membership rows:", len(df))
print("Confirmed near-duplicate edges:", len(confirmed))

required_df_columns = [
    'path',
    'class',
    'filename',
    'sha256',
    'global_family_id',
    'mixed_label_family'
]

for col in required_df_columns:
    assert col in df.columns, f"STOP: Missing column: {col}"

assert 'path_a' in confirmed.columns, "STOP: path_a missing"
assert 'path_b' in confirmed.columns, "STOP: path_b missing"

# Fix boolean column if CSV loaded True/False as text
if df['mixed_label_family'].dtype == object:

    df['mixed_label_family'] = (
        df['mixed_label_family']
        .astype(str)
        .str.strip()
        .str.lower()
        .map({
            'true': True,
            'false': False
        })
    )

assert df['mixed_label_family'].notna().all(), (
    "STOP: mixed_label_family contains invalid values."
)

print(
    "Mixed-label images:",
    int(df['mixed_label_family'].sum())
)

print("\n✅ CELL 2 COMPLETE")

Checking saved analysis files...
global_family_membership.csv: True
GLOBAL_CONFIRMED_NEAR_DUPLICATE_EDGES.csv: True

Loaded saved analysis.
Family membership rows: 13624
Confirmed near-duplicate edges: 1490
Mixed-label images: 14

✅ CELL 2 COMPLETE


## Phase 2 — Global all-pairs pHash candidate generation

This is the key improvement.

Every image is compared against every later image, not merely its nearest neighbour.
Only pairs with pHash Hamming distance ≤ 12 are kept for expensive SIFT verification.

For ~13.6k images this is ~93 million cheap hash comparisons, performed in vectorized blocks.

In [4]:
# ============================================================
# CELL 3 — REBUILD FAMILY-LEVEL SPLIT ASSIGNMENT
# ============================================================

eligible = df[
    ~df['mixed_label_family']
].copy()

fam = (
    eligible
    .groupby('global_family_id')
    .agg(
        class_name=('class', 'first'),
        n_files=('path', 'size')
    )
    .reset_index()
)

TARGET = {
    'Train': 0.65,
    'Validation': 0.16,
    'Test': 0.19
}

rng = np.random.default_rng(SEED)

assignment = {}

print("Rebuilding family-level Train/Validation/Test assignment...\n")

for cls in CLASS_ORDER:

    fc = fam[
        fam['class_name'] == cls
    ].copy()

    fc['rand'] = rng.random(len(fc))

    fc = fc.sort_values(
        ['n_files', 'rand'],
        ascending=[False, True]
    )

    total = int(fc['n_files'].sum())

    targets = {
        k: TARGET[k] * total
        for k in TARGET
    }

    counts = {
        k: 0
        for k in TARGET
    }

    for _, row in fc.iterrows():

        deficits = {
            k: (
                targets[k] - counts[k]
            ) / max(targets[k], 1)
            for k in TARGET
        }

        destination = max(
            deficits,
            key=deficits.get
        )

        family_id = row['global_family_id']

        assignment[family_id] = destination

        counts[destination] += int(
            row['n_files']
        )

    print(
        cls,
        "Total =", total,
        "| Split =", counts
    )

print(
    "\nNumber of assigned families:",
    len(assignment)
)

print("\n✅ CELL 3 COMPLETE")

Rebuilding family-level Train/Validation/Test assignment...

Cataract Total = 4494 | Split = {'Train': 2921, 'Validation': 719, 'Test': 854}
Normal Total = 5141 | Split = {'Train': 3341, 'Validation': 823, 'Test': 977}
Not Eye Total = 3975 | Split = {'Train': 2583, 'Validation': 636, 'Test': 756}

Number of assigned families: 12201

✅ CELL 3 COMPLETE


## Phase 3 — Global SIFT + RANSAC verification

Strict confirmation rule (same as earlier audits):

- at least 20 Lowe-ratio SIFT matches
- at least 15 RANSAC inliers
- inlier ratio ≥ 0.50

Progress is saved every 250 pairs, so the notebook can resume after a disconnect.

In [5]:
# ============================================================
# CELL 4 — DELETE ONLY INCOMPLETE FINAL OUTPUT
# ============================================================

expected_final = Path(
    '/content/drive/MyDrive/Cataract/'
    'Data_Clean_LeakageControlled_FINAL'
)

expected_quar = Path(
    '/content/drive/MyDrive/Cataract/'
    'FINAL_REVISION_2026_08/'
    'global_family_quarantine'
)

# Safety checks
assert FINAL == expected_final, (
    f"STOP: Unexpected FINAL path: {FINAL}"
)

assert QUAR == expected_quar, (
    f"STOP: Unexpected quarantine path: {QUAR}"
)

# Extra safety — NEVER allow deletion of SOURCE
assert FINAL != SOURCE
assert 'Data_Clean_LeakageControlled_v2' not in FINAL.name

print("We will remove ONLY:")
print(FINAL)
print(QUAR)

if FINAL.exists():

    print("\nRemoving incomplete FINAL folder...")

    shutil.rmtree(FINAL)

    print("✅ Incomplete FINAL folder removed")

else:

    print("\nFINAL folder was not present. That is okay.")

if QUAR.exists():

    print("Removing incomplete quarantine folder...")

    shutil.rmtree(QUAR)

    print("✅ Incomplete quarantine folder removed")

else:

    print("Quarantine folder was not present. That is okay.")

QUAR.mkdir(
    parents=True,
    exist_ok=True
)

print("\nOriginal V2 source still exists:", SOURCE.exists())

assert SOURCE.exists()

print("\n✅ CELL 4 COMPLETE")

We will remove ONLY:
/content/drive/MyDrive/Cataract/Data_Clean_LeakageControlled_FINAL
/content/drive/MyDrive/Cataract/FINAL_REVISION_2026_08/global_family_quarantine

Removing incomplete FINAL folder...
✅ Incomplete FINAL folder removed
Removing incomplete quarantine folder...
✅ Incomplete quarantine folder removed

Original V2 source still exists: True

✅ CELL 4 COMPLETE


## Phase 4 — Build complete connected image families

Edges used:
- all exact SHA-256 duplicates
- all globally confirmed SIFT/RANSAC near-duplicate edges

Connected components create full image families transitively.

In [6]:
# ============================================================
# CELL 5 — CREATE FINAL DATASET + SAVE MANIFEST
# ============================================================

# ------------------------------------------------------------
# A. Create destination folders
# ------------------------------------------------------------

for split in SPLITS:

    for cls in CLASS_ORDER:

        (
            FINAL
            / split
            / cls
        ).mkdir(
            parents=True,
            exist_ok=True
        )

QUAR.mkdir(
    parents=True,
    exist_ok=True
)

print("✅ FINAL directory structure created")

# ------------------------------------------------------------
# B. Prepare deterministic destination names
# ------------------------------------------------------------

work = df.copy()

work['planned_split'] = work.apply(
    lambda r:
        'Quarantine'
        if r['mixed_label_family']
        else assignment[r['global_family_id']],
    axis=1
)

# Detect filenames that would collide inside the same destination
collision_counts = (
    work
    .groupby([
        'planned_split',
        'class',
        'filename'
    ])['path']
    .transform('count')
)

work['filename_collision'] = collision_counts > 1

print(
    "Potential destination filename collisions:",
    int(work['filename_collision'].sum())
)

# ------------------------------------------------------------
# C. Copy files
# ------------------------------------------------------------

manifest_rows = []

total_rows = len(work)

print("\nStarting image copy...")
print("Total source rows:", total_rows)
print("This can take several minutes.\n")

for i, (_, row) in enumerate(
    work.iterrows(),
    start=1
):

    source_path = SOURCE / row['path']

    assert source_path.exists(), (
        f"STOP: Missing source image: {source_path}"
    )

    cls = row['class']

    family_id = row['global_family_id']

    original_filename = row['filename']

    stem = Path(original_filename).stem
    suffix = Path(original_filename).suffix

    # Use deterministic suffix only if duplicate filename collision exists
    if row['filename_collision']:

        short_id = hashlib.sha1(
            row['path'].encode('utf-8')
        ).hexdigest()[:8]

        output_filename = (
            f"{stem}__{family_id}__{short_id}{suffix}"
        )

    else:

        output_filename = original_filename

    if row['mixed_label_family']:

        split = 'Quarantine'

        destination = (
            QUAR
            / family_id
            / cls
            / output_filename
        )

        destination.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        status = 'QUARANTINED_MIXED_LABEL_FAMILY'

    else:

        split = assignment[family_id]

        destination = (
            FINAL
            / split
            / cls
            / output_filename
        )

        status = 'INCLUDED'

    shutil.copy2(
        source_path,
        destination
    )

    manifest_rows.append({

        'source_v2_path':
            row['path'],

        'class':
            cls,

        'global_family_id':
            family_id,

        'final_split':
            split,

        'status':
            status,

        'original_filename':
            original_filename,

        'output_filename':
            output_filename,

        'final_path':
            str(destination)
    })

    if i % 1000 == 0:

        print(
            f"Copied {i} / {total_rows}"
        )

# ------------------------------------------------------------
# D. Create manifest
# ------------------------------------------------------------

manifest = pd.DataFrame(
    manifest_rows
)

manifest_path = (
    REPORT
    / 'FINAL_split_manifest.csv'
)

manifest.to_csv(
    manifest_path,
    index=False
)

print("\n======================================")
print("✅ PHASE 6 REBUILD COMPLETED")
print("======================================")

print(
    "Manifest saved at:",
    manifest_path
)

print(
    "Manifest rows:",
    len(manifest)
)

print(
    "Included images:",
    int(
        (
            manifest['status']
            == 'INCLUDED'
        ).sum()
    )
)

print(
    "Quarantined images:",
    int(
        (
            manifest['status']
            == 'QUARANTINED_MIXED_LABEL_FAMILY'
        ).sum()
    )
)

print("\n✅ CELL 5 COMPLETE")

✅ FINAL directory structure created
Potential destination filename collisions: 0

Starting image copy...
Total source rows: 13624
This can take several minutes.

Copied 1000 / 13624
Copied 2000 / 13624
Copied 3000 / 13624
Copied 4000 / 13624
Copied 5000 / 13624
Copied 6000 / 13624
Copied 7000 / 13624
Copied 8000 / 13624
Copied 9000 / 13624
Copied 10000 / 13624
Copied 11000 / 13624
Copied 12000 / 13624
Copied 13000 / 13624

✅ PHASE 6 REBUILD COMPLETED
Manifest saved at: /content/drive/MyDrive/Cataract/FINAL_REVISION_2026_08/global_family_audit/FINAL_split_manifest.csv
Manifest rows: 13624
Included images: 13610
Quarantined images: 14

✅ CELL 5 COMPLETE


## Phase 5 — Family-level Train / Validation / Test split

Target proportions:
- Train ~65%
- Validation ~16%
- Test ~19%

An entire global family can exist in only one partition.
Mixed-label families are quarantined.

In [7]:
# ============================================================
# CELL 6 — VERIFY FINAL DATASET AND MANIFEST
# ============================================================

manifest_path = (
    REPORT
    / 'FINAL_split_manifest.csv'
)

print(
    "Manifest exists:",
    manifest_path.exists()
)

assert manifest_path.exists(), (
    "STOP: Manifest was not saved."
)

manifest = pd.read_csv(
    manifest_path
)

print(
    "Manifest rows:",
    len(manifest)
)

print("\nSTATUS COUNTS:")

print(
    manifest
    .groupby([
        'status',
        'class'
    ])
    .size()
    .unstack(fill_value=0)
)

print("\nFINAL SPLIT COUNTS:")

final_counts = (
    manifest[
        manifest['status']
        == 'INCLUDED'
    ]
    .groupby([
        'final_split',
        'class'
    ])
    .size()
    .unstack(fill_value=0)
)

print(final_counts)

# Count actual files inside FINAL directory
actual_final_files = 0

for split in SPLITS:

    for cls in CLASS_ORDER:

        actual_final_files += sum(
            1
            for p in (
                FINAL / split / cls
            ).iterdir()
            if p.is_file()
        )

expected_final_files = int(
    (
        manifest['status']
        == 'INCLUDED'
    ).sum()
)

print("\nExpected FINAL files:", expected_final_files)
print("Actual FINAL files  :", actual_final_files)

assert actual_final_files == expected_final_files, (
    "STOP: Actual FINAL file count does not match manifest."
)

print("\n✅ FINAL file count matches manifest")
print("✅ CELL 6 COMPLETE")

Manifest exists: True
Manifest rows: 13624

STATUS COUNTS:
class                           Cataract  Normal  Not Eye
status                                                   
INCLUDED                            4494    5141     3975
QUARANTINED_MIXED_LABEL_FAMILY         8       5        1

FINAL SPLIT COUNTS:
class        Cataract  Normal  Not Eye
final_split                           
Test              854     977      756
Train            2921    3341     2583
Validation        719     823      636

Expected FINAL files: 13610
Actual FINAL files  : 13610

✅ FINAL file count matches manifest
✅ CELL 6 COMPLETE


## Phase 6 — Write FINAL dataset

This creates a new folder. It does not overwrite V2.

If the FINAL folder already exists, the notebook stops to prevent accidental duplication.

In [8]:
# ============================================================
# CELL 7 — PHASE 7 INTERNAL LEAKAGE CHECK
# ============================================================

# Reload the files from Drive for extra safety

manifest = pd.read_csv(
    REPORT / 'FINAL_split_manifest.csv'
)

confirmed = pd.read_csv(
    REPORT / 'GLOBAL_CONFIRMED_NEAR_DUPLICATE_EDGES.csv'
)

df = pd.read_csv(
    REPORT / 'global_family_membership.csv'
)

print("Running Phase 7 checks...\n")

# ------------------------------------------------------------
# CHECK 1 — Confirmed near-duplicate edges
# ------------------------------------------------------------

m = manifest.set_index(
    'source_v2_path'
)

bad_edges = []

for _, row in confirmed.iterrows():

    path_a = row['path_a']
    path_b = row['path_b']

    if path_a not in m.index:
        raise RuntimeError(
            f"STOP: Missing path from manifest: {path_a}"
        )

    if path_b not in m.index:
        raise RuntimeError(
            f"STOP: Missing path from manifest: {path_b}"
        )

    split_a = m.loc[
        path_a,
        'final_split'
    ]

    split_b = m.loc[
        path_b,
        'final_split'
    ]

    if (
        split_a in SPLITS
        and
        split_b in SPLITS
        and
        split_a != split_b
    ):

        bad_edges.append({
            'path_a': path_a,
            'path_b': path_b,
            'split_a': split_a,
            'split_b': split_b
        })

print(
    "Confirmed global edges crossing FINAL split:",
    len(bad_edges)
)

# ------------------------------------------------------------
# CHECK 2 — Exact SHA duplicates
# ------------------------------------------------------------

tmp = (
    df[
        [
            'path',
            'sha256'
        ]
    ]
    .merge(
        manifest[
            [
                'source_v2_path',
                'final_split'
            ]
        ],
        left_on='path',
        right_on='source_v2_path',
        how='inner'
    )
)

bad_exact = []

for sha, group in (
    tmp[
        tmp[
            'final_split'
        ].isin(SPLITS)
    ]
    .groupby('sha256')
):

    if (
        group[
            'final_split'
        ].nunique()
        > 1
    ):

        bad_exact.append(group)

print(
    "Exact SHA groups crossing FINAL split:",
    len(bad_exact)
)

# ------------------------------------------------------------
# SAVE SUMMARY
# ------------------------------------------------------------

included_images = int(
    (
        manifest['status']
        == 'INCLUDED'
    ).sum()
)

quarantined_images = int(
    (
        manifest['status']
        == 'QUARANTINED_MIXED_LABEL_FAMILY'
    ).sum()
)

summary = {

    'input_v2_images':
        int(len(df)),

    'manifest_rows':
        int(len(manifest)),

    'final_included_images':
        included_images,

    'quarantined_images':
        quarantined_images,

    'confirmed_edges_crossing_final':
        int(len(bad_edges)),

    'exact_sha_groups_crossing_final':
        int(len(bad_exact))
}

summary_path = (
    REPORT
    / 'FINAL_phase7_summary.json'
)

with open(
    summary_path,
    'w'
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )

print("\n==============================")
print("PHASE 7 SUMMARY")
print("==============================")

print(
    json.dumps(
        summary,
        indent=2
    )
)

# ------------------------------------------------------------
# DECISION
# ------------------------------------------------------------

if (
    len(bad_edges) == 0
    and
    len(bad_exact) == 0
):

    print("\n======================================")
    print("✅ PHASE 7 PASSED")
    print("======================================")

    print(
        "01E global clustering is complete."
    )

    print(
        "DO NOT TRAIN MODELS YET."
    )

    print(
        "Next step will be 01F independent verification."
    )

else:

    print("\n======================================")
    print("❌ PHASE 7 FAILED")
    print("======================================")

    print(
        "Do NOT start model training."
    )

print("\n✅ CELL 7 COMPLETE")

Running Phase 7 checks...

Confirmed global edges crossing FINAL split: 0
Exact SHA groups crossing FINAL split: 0

PHASE 7 SUMMARY
{
  "input_v2_images": 13624,
  "manifest_rows": 13624,
  "final_included_images": 13610,
  "quarantined_images": 14,
  "confirmed_edges_crossing_final": 0,
  "exact_sha_groups_crossing_final": 0
}

✅ PHASE 7 PASSED
01E global clustering is complete.
DO NOT TRAIN MODELS YET.
Next step will be 01F independent verification.

✅ CELL 7 COMPLETE
